# NLP Day 1 · 실습 2. BoW와 TF-IDF로 영화 리뷰 분류하기

**연결 이론:** 이론 3 — 문서 벡터, BoW, TF-IDF, 텍스트 분류, 데이터 누수

## 1. 이번 실습에서 만들 것

영화 리뷰를 숫자 벡터로 바꾸고, **같은 분류 모델에서 BoW와 TF-IDF 표현을 비교**합니다.

- **A형 함께 구현하기:** 작은 문서–단어 행렬 확인 → BoW 변환 → Logistic Regression 학습 → Accuracy·F1 평가
- **B형 직접 완성하기:** `make_tfidf_features()`에서 `fit_transform()`과 `transform()` 연결 → 동일한 분류기로 결과 비교
- **결과 해석:** 오분류 리뷰를 살펴보고 **학습 데이터에서만 어휘집과 IDF를 학습하는 이유**를 설명합니다.

**데이터 안내:** 공식 [NSMC (Naver Sentiment Movie Corpus)](https://github.com/e9t/nsmc)의 `ratings_train.txt`에서 고정된 훈련/검증 부분집합을 준비합니다. `0=부정`, `1=긍정`입니다. 최초 실행 시 다운로드하여 `assets/nlp/nsmc_train_subset.csv`, `assets/nlp/nsmc_val_subset.csv`로 저장하며, 다음 수업에서도 같은 파일을 사용할 수 있습니다. 이미 저장된 경우 다시 만들지 않습니다.

인터넷을 사용할 수 없는 경우에는 ZIP에 포함된 **별도의 인공 연습 데이터**(`demo_*.csv`)를 사용합니다. 이 자료는 NSMC가 아니므로 실제 NSMC 성능으로 해석하지 않습니다. 원본 데이터에 대한 정보는 [NSMC 저장소](https://github.com/e9t/nsmc)를 참고하세요.

실행에 필요한 라이브러리: `pandas`, `scikit-learn` (또는 기존 과정의 동일한 Python 환경).

## 2. A형 · 함께 구현하기

### 2.1 데이터 경로와 학습·검증 자료 준비

노트북은 기존 CV 실습 파일과 **같은 프로젝트 루트**에 놓고 실행합니다. 데이터는 `assets/nlp/`에서 읽고, 실행 결과는 `outputs/nlp/`에 저장합니다.

In [ ]:
from pathlib import Path
from io import BytesIO
from urllib.request import urlopen
import json
import re

import pandas as pd

# 학습 데이터와 검증 데이터를 분리할 때 사용
from sklearn.model_selection import train_test_split

# BoW와 TF-IDF 벡터화에 사용할 도구
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# 영화 리뷰의 긍정·부정을 분류할 모델
from sklearn.linear_model import LogisticRegression

# 분류 모델의 성능을 평가할 지표
from sklearn.metrics import accuracy_score, f1_score

# [실습 데이터와 결과 저장경로 지정]
# 현재 작업 폴더를 프로젝트 루트로 사용
ROOT = Path.cwd()
# 데이터 파일이 저장되는 폴더
DATA_DIR = ROOT / "assets" / "nlp"
# 실습 결과가 저장되는 폴더
OUTPUT_DIR = ROOT / "outputs" / "nlp"
# 결과 폴더가 없으면 자동 생성
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 데이터 폴더가 없으면 안내 오류 발생
if not DATA_DIR.exists():
    raise FileNotFoundError("assets/nlp 폴더를 찾을 수 없습니다. ZIP에 포함된 assets/nlp 폴더를 프로젝트 루트에 풀어주세요.")

# [사용할 데이터 파일 경로 지정]
NSMC_TRAIN = DATA_DIR / "nsmc_train_subset.csv"
NSMC_VAL = DATA_DIR / "nsmc_val_subset.csv"
DEMO_TRAIN = DATA_DIR / "demo_train.csv"
DEMO_VAL = DATA_DIR / "demo_val.csv"
NSMC_URL = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"

# [데이터 준비 함수]
def prepare_data():
    # 이미 만들어진 NSMC 부분집합은 다시 샘플링하지 않습니다.
    if NSMC_TRAIN.exists() and NSMC_VAL.exists():
        return pd.read_csv(NSMC_TRAIN), pd.read_csv(NSMC_VAL), "NSMC 부분집합"

    try:
        print("NSMC 원본에서 고정된 부분집합을 준비합니다...")
        # NSMC 원본 데이터를 읽어옴
        with urlopen(NSMC_URL, timeout=10) as response:
            original = pd.read_csv(BytesIO(response.read()), sep="\t")
        # 문서나 정답이 비어 있는 데이터 제거    
        original = original.dropna(subset=["document", "label"]).copy()
        # 문서를 문자열로 변환하고 앞뒤 공백 제거
        original["document"] = original["document"].astype(str).str.strip()
        # 내용이 없는 문서 제거
        original = original[original["document"].ne("")]
        # 중복 문서 제거
        original = original.drop_duplicates(subset=["document"])
        # 부정(0), 긍정(1) 리뷰를 각각 1,250개씩 선택
        # random_state를 고정하여 같은 데이터가 선택되도록 함
        selected = pd.concat([
            original[original["label"] == label].sample(n=1250, random_state=2026)
            for label in (0, 1)
        ], ignore_index=True)

        # 선택한 데이터를 학습 80%, 검증 20%로 분리
        # stratify: 두 데이터의 긍정·부정 비율을 유지
        train_df, val_df = train_test_split(
            selected, test_size=0.20, random_state=42, stratify=selected["label"]
        )

        # 행 인덱스 정리
        train_df = train_df.reset_index(drop=True)
        val_df = val_df.reset_index(drop=True)

        # 만들어진 학습·검증 데이터를 CSV로 저장
        train_df.to_csv(NSMC_TRAIN, index=False, encoding="utf-8-sig")
        val_df.to_csv(NSMC_VAL, index=False, encoding="utf-8-sig")
        print("NSMC 파일 저장 완료:", NSMC_TRAIN.name, NSMC_VAL.name)
        return train_df, val_df, "NSMC 부분집합"
    except Exception as exc:
        print("NSMC 다운로드에 실패하여 오프라인 인공 연습 데이터를 사용합니다.")
        print("오류 유형:", type(exc).__name__)
        if not (DEMO_TRAIN.exists() and DEMO_VAL.exists()):
            raise FileNotFoundError("오프라인 연습용 CSV를 찾을 수 없습니다. assets/nlp 폴더를 확인하세요.") from exc
        return pd.read_csv(DEMO_TRAIN), pd.read_csv(DEMO_VAL), "오프라인 인공 연습 데이터 (NSMC 아님)"

# [데이터 불러오기]
train_df, val_df, data_source = prepare_data()
# 학습·검증 데이터의 기본 형식 정리
for df in (train_df, val_df):
    # 문서나 정답이 비어 있는 행 제거
    df.dropna(subset=["document", "label"], inplace=True)
    # 리뷰는 문자열, 정답은 정수 형태로 변환
    df["document"] = df["document"].astype(str)
    df["label"] = df["label"].astype(int)

# [데이터 검증 및 요약]
# 학습·검증 데이터에 부정(0), 긍정(1)이 모두 있는지 확인
assert set(train_df["label"]) == {0, 1}
assert set(val_df["label"]) == {0, 1}
# 학습과 검증 데이터에 동일한 리뷰 ID가 없는지 확인
assert set(train_df["id"]).isdisjoint(set(val_df["id"]))
print("자료 유형:", data_source)
print("학습/검증 건수:", len(train_df), "/", len(val_df))
print("학습 라벨 분포:", train_df["label"].value_counts().sort_index().to_dict())
print("검증 라벨 분포:", val_df["label"].value_counts().sort_index().to_dict())

NSMC 원본에서 고정된 부분집합을 준비합니다...
NSMC 파일 저장 완료: nsmc_train_subset.csv nsmc_val_subset.csv
자료 유형: NSMC 부분집합
학습/검증 건수: 2000 / 500
학습 라벨 분포: {0: 1000, 1: 1000}
검증 라벨 분포: {0: 250, 1: 250}


C:\Users\kjm05\AppData\Local\Temp\ipykernel_5976\98983978.py:64: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.dropna(subset=["document", "label"], inplace=True)
C:\Users\kjm05\AppData\Local\Temp\ipykernel_5976\98983978.py:64: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.dropna(subset=["document", "label"], inplace=True)


### 2.2 짧은 리뷰로 BoW의 행과 열 확인하기

실습의 첫 단계에서는 문서가 행, 어휘가 열인 **문서–단어 행렬**을 직접 확인합니다. 문장을 나누는 방법은 복잡한 형태소 분석 대신 간단한 공백 기준을 사용합니다.

In [ ]:
# 텍스트 정규화 함수
def normalize_text(text):

    # 문장부호를 공백으로 변경
    # 예: "재미있다." → "재미있다 "
    text = re.sub(r"[^\w\s]", " ", str(text))
    # 연속된 공백을 하나로 만들고 앞뒤 공백 제거
    return re.sub(r"\s+", " ", text).strip()

# 예제로 사용할 영화 리뷰 4개
sample_reviews = [
    "정말 재미있다.",
    "연기가 좋고 재미있다.",
    "내용이 지루하다.",
    "재미있을 줄 알았는데 지루하다.",
]

# BoW 벡터 변환기 생성
sample_vectorizer = CountVectorizer(
    tokenizer=str.split,  # 공백 기준으로 토큰화
    token_pattern=None,   # 직접 지정한 토큰화 방식 사용
    lowercase=False)    # 대소문자를 자동으로 변경하지 않음

# 리뷰를 BoW 숫자 행렬로 변환
# fit_transform(): 리뷰에 등장하는 단어들로 어휘집을 만들고,각 리뷰의 단어 등장 횟수를 숫자로 변환
sample_matrix = sample_vectorizer.fit_transform([normalize_text(x) for x in sample_reviews])

# 결과를 보기 쉬운 표로 변환
sample_dtm = pd.DataFrame(

    # BoW 행렬을 일반 배열로 변환
    sample_matrix.toarray(),

    # 각 행은 하나의 리뷰를 나타냄
    index=[f"리뷰 {i}" for i in range(1, len(sample_reviews) + 1)],

    # 각 열은 어휘집에 등록된 단어를 나타냄
    columns=sample_vectorizer.get_feature_names_out(),
)

# BoW 행렬 결과 확인
print("행렬 Shape:", sample_matrix.shape)
print("행 = 리뷰, 열 = 어휘, 값 = 등장 횟수")
display(sample_dtm)

행렬 Shape: (4, 9)
행 = 리뷰, 열 = 어휘, 값 = 등장 횟수


,내용이,알았는데,연기가,재미있다,재미있을,정말,좋고,줄,지루하다
리뷰 1,0,0,0,1,0,1,0,0,0
리뷰 2,0,0,1,1,0,0,1,0,0
리뷰 3,1,0,0,0,0,0,0,0,1
리뷰 4,0,1,0,0,1,0,0,1,1


### 2.3 NSMC 자료를 BoW로 변환하고 분류하기

**중요:** 어휘집은 **학습 리뷰로만** 만들고 검증 리뷰에는 같은 변환기를 적용합니다. 학습/검증 데이터에서 각각 `fit_transform()`을 실행하지 않습니다.

In [ ]:
# [학습·검증 리뷰 전처리]
# 앞에서 만든 정규화 함수를 양쪽 데이터에 동일하게 적용
train_df["clean_text"] = train_df["document"].map(normalize_text)
val_df["clean_text"] = val_df["document"].map(normalize_text)

# 전처리 후 빈 문장은 제거
# 모델에 입력할 수 있는 정상적인 문장만 남김
train_df = train_df[train_df["clean_text"].ne("")].copy().reset_index(drop=True)
val_df = val_df[val_df["clean_text"].ne("")].copy().reset_index(drop=True)


# [모델 입력 데이터와 정답 분리]
# X: 모델에 입력할 영화 리뷰 텍스트
X_train_text = train_df["clean_text"]
X_val_text = val_df["clean_text"]
# y: 리뷰의 실제 정답 (0=부정, 1=긍정)
y_train = train_df["label"]
y_val = val_df["label"]

# BoW 벡터 변환기 생성
bow_vectorizer = CountVectorizer(
    tokenizer=str.split, token_pattern=None, lowercase=False,
    min_df=2,  # 최소 2개 문서에서 등장한 단어 사용
    max_features=8000, # 사용할 어휘 수를 최대 8,000개로 제한
)

# [학습·검증 리뷰를 BoW 벡터로 변환]
# 학습 데이터에서 어휘집을 만들고 BoW 벡터 생성
X_train_bow = bow_vectorizer.fit_transform(X_train_text)
# 검증 데이터는 학습한 어휘집으로만 변환
# 검증 데이터를 이용해 어휘집을 새로 만들지 않음  
X_val_bow = bow_vectorizer.transform(X_val_text)          

# [만들어진 BoW 행렬 확인]
print("BoW 학습 Shape:", X_train_bow.shape)
print("BoW 검증 Shape:", X_val_bow.shape)
# 학습 데이터로 만든 어휘집의 단어 개수
print("학습한 어휘 수:", len(bow_vectorizer.vocabulary_))

# 학습·검증 행렬의 열 개수가 같은지 확인
# 같은 어휘집을 사용했다면 열 개수도 같아야 함
assert X_train_bow.shape[1] == X_val_bow.shape[1]

# 영화 리뷰 분류 모델 생성
# Logistic Regression:입력된 BoW 벡터를 이용해 긍정·부정을 분류하는 모델
bow_model = LogisticRegression(max_iter=1000,  # 모델 학습 시 최대 반복 횟수 (1000번)
                               random_state=42, # 난수 시드 고정 (재현성 확보)
                               solver="liblinear") # 모델의 가중치를 최적화하는 알고리즘 지정 (작은 데이터 분류셋에 적합)

# 학습 리뷰의 BoW 벡터와 실제 정답을 이용해 학습
bow_model.fit(X_train_bow, y_train)
# 학습이 끝난 모델로 검증 리뷰의 긍정·부정을 예측
bow_preds = bow_model.predict(X_val_bow)

# 분류 성능 평가 함수
def evaluate_predictions(y_true, y_pred):
    return {
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "F1": float(f1_score(y_true, y_pred, pos_label=1)),
    }

# 실제 정답과 모델 예측 결과를 비교
bow_scores = evaluate_predictions(y_val, bow_preds)
print("BoW 검증 결과:", {k: round(v, 4) for k, v in bow_scores.items()})

(OUTPUT_DIR / "day1_bow_metrics.json").write_text(
    json.dumps({"source": data_source, **bow_scores}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

BoW 학습 Shape: (2000, 1567)
BoW 검증 Shape: (500, 1567)
학습한 어휘 수: 1567
BoW 검증 결과: {'Accuracy': 0.648, 'F1': 0.614}


76

### 2.4 분류 결과와 오분류 사례 확인하기

지표뿐 아니라 실제 리뷰를 함께 살펴보면 모델이 **어떤 표현을 잘못 판단했는지** 확인할 수 있습니다.

In [ ]:
# [예측 결과와 실제 정답 비교]
# bow_preds: 모델이 예측한 긍정(1)·부정(0) 결과
# y_val: 검증 데이터의 실제 정답
# != : 두 값이 서로 다른지 비교
# 예측과 정답이 다르면 True, 같으면 False
error_mask = bow_preds != y_val.to_numpy()

error_examples = val_df.loc[error_mask, ["document", "label"]].copy()
error_examples["예측"] = bow_preds[error_mask]
print("BoW 오분류 개수:", len(error_examples))
if len(error_examples):
    display(error_examples.head(3).rename(columns={"document": "리뷰", "label": "정답"}))
else:
    print("이번 검증 자료에서 오분류가 없었습니다.")

BoW 오분류 개수: 176


,리뷰,정답,예측
1,마지막 장면을 잊을 수가 없다,1,0
4,김민 죽어버려 성룡을망치다니,1,0
6,볼려다가 평점 때문에 안볼란다밑에 4명 티안나게 좀 알바하자!,0,1


## 3. B형 · 직접 완성하기

### 3.1 응용 문제: BoW를 TF-IDF로 바꾸기

같은 영화 리뷰와 같은 분류 모델을 사용하되, **입력 표현만 TF-IDF로 변경**합니다.

**과제:** `make_tfidf_features()`에서 다음 두 부분을 완성하세요.

- 학습 문장에 대해 `vectorizer.fit_transform(...)` 실행
- 검증 문장에 대해 **이미 학습한** `vectorizer.transform(...)` 실행

`TfidfVectorizer`의 기본 동작에는 벡터 길이를 정규화하는 과정 등이 포함되므로, 수업에서 직접 계산한 TF-IDF 숫자와 출력값이 완전히 같지는 않을 수 있습니다.

In [5]:
def make_tfidf_features(train_texts, val_texts):
    vectorizer = TfidfVectorizer(
        tokenizer=str.split, token_pattern=None, lowercase=False,
        min_df=2, max_features=8000,
    )

    # TODO 1: train_texts로 변환기를 학습하고 학습 문서 벡터를 만드세요.
    X_train_tfidf = None

    # TODO 2: 학습된 같은 변환기로 검증 문서 벡터를 만드세요.
    X_val_tfidf = None

    return vectorizer, X_train_tfidf, X_val_tfidf


tfidf_vectorizer, X_train_tfidf, X_val_tfidf = make_tfidf_features(X_train_text, X_val_text)
print("B형 변환 상태:", "완료" if X_train_tfidf is not None and X_val_tfidf is not None else "TODO 미완성")

B형 변환 상태: TODO 미완성


### 3.2 같은 분류기로 결과 비교하기

아래 셀은 B형의 `TODO`를 완료한 뒤 다시 실행하면 TF-IDF의 성능을 측정하고 비교표를 만들어 줍니다. **TF-IDF 성능이 반드시 BoW보다 높아야 하는 것은 아닙니다.**

In [6]:
comparison = pd.DataFrame([{"표현 방식": "BoW", **bow_scores}])

if X_train_tfidf is None or X_val_tfidf is None:
    print("TODO가 아직 완성되지 않았습니다. TF-IDF 변환 코드를 작성하고 다시 실행하세요.")
else:
    assert X_train_tfidf.shape[0] == len(train_df)
    assert X_val_tfidf.shape[0] == len(val_df)
    assert X_train_tfidf.shape[1] == X_val_tfidf.shape[1]

    tfidf_model = LogisticRegression(max_iter=1000, random_state=42, solver="liblinear")
    tfidf_model.fit(X_train_tfidf, y_train)
    tfidf_preds = tfidf_model.predict(X_val_tfidf)
    tfidf_scores = evaluate_predictions(y_val, tfidf_preds)
    comparison = pd.DataFrame([
        {"표현 방식": "BoW", **bow_scores},
        {"표현 방식": "TF-IDF", **tfidf_scores},
    ])
    print("TF-IDF 오분류 예시:")
    tfidf_errors = val_df.loc[tfidf_preds != y_val.to_numpy(), ["document", "label"]].copy()
    tfidf_errors["예측"] = tfidf_preds[tfidf_preds != y_val.to_numpy()]
    if len(tfidf_errors):
        display(tfidf_errors.head(3).rename(columns={"document": "리뷰", "label": "정답"}))
    else:
        print("이번 검증 자료에서 오분류가 없었습니다.")

    # 이후 실습에서 동일한 성능 비교를 재사용할 수 있습니다.
    comparison.to_csv(OUTPUT_DIR / "day1_bow_tfidf_comparison.csv", index=False, encoding="utf-8-sig")

print("자료 유형:", data_source)
display(comparison.round(4))

TODO가 아직 완성되지 않았습니다. TF-IDF 변환 코드를 작성하고 다시 실행하세요.
자료 유형: NSMC 부분집합


,표현 방식,Accuracy,F1
0,BoW,0.648,0.614


## 4. 결과 확인과 이번 실습 간단하게 요약

**확인할 결과:** BoW와 TF-IDF의 Accuracy·F1, 문서 벡터의 Shape, 잘못 분류한 영화 리뷰. B형 코드를 완성해 비교표에 두 행이 나타나는지 확인하세요.

**첫째,** BoW는 문서에 등장한 단어들의 횟수로 문서를 벡터화합니다.

**둘째,** TF-IDF는 단어 빈도와 문서 전체에서의 등장 분포를 함께 반영하여 가중치를 계산합니다.

**셋째,** 학습 데이터에서 만든 어휘집과 IDF를 **검증 데이터에 그대로 적용**해야 올바르게 평가할 수 있습니다.

**넷째,** BoW와 TF-IDF를 비교할 때는 같은 데이터와 같은 분류 모델을 사용하고, 결과의 차이를 실제 지표와 오분류 사례를 근거로 설명해야 합니다.

**다섯째,** 이러한 표현은 단어의 등장 정보를 다루지만 문맥과 단어 순서를 직접 표현하지 못합니다. 다음 실습에서는 **단어의 등장 순서**를 Bigram으로 표현해 다음 단어를 예측합니다.

> 저장된 `assets/nlp/nsmc_train_subset.csv`와 `nsmc_val_subset.csv`가 있다면, 이후 BERT 분류 실습에서도 **같은 분할**을 사용해 비교할 수 있습니다. 현재 자료 유형이 오프라인 연습 데이터라면 결과를 NSMC 실험 결과로 표기하지 않습니다.